In [3]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Desabilitar verificação SSL (solução para erro de certificado)
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

# Configurações
os.makedirs("resultados_transformer_preco", exist_ok=True)
os.makedirs("dados_historicos_salvos", exist_ok=True)  # Pasta para salvar dados

# Lista de tickers
tickers = ["BEEF3.SA", "BRFS3.SA", "VALE3.SA", "JBSS3.SA", 
           "SOJA3.SA", "SUZB3.SA", "GGBR3.SA", "CSNA3.SA"]

# Janelas temporais para experimentos
janelas_temporais = [5, 10, 15, 20]

# Data de corte para treino/teste
DATA_CORTE = "2023-01-01"

print("="*80)
print(" "*20 + "TRANSFORMER - APENAS PREÇOS HISTÓRICOS")
print("="*80)
print(f"Tickers: {', '.join(tickers)}")
print(f"Janelas Temporais: {janelas_temporais}")
print(f"Data de Corte Treino/Teste: {DATA_CORTE}")
print("="*80 + "\n")

# =============================================================================
# FUNÇÕES AUXILIARES
# =============================================================================

def baixar_dados_historicos(ticker, start_date="2010-01-01", end_date="2024-12-31"):
    """Baixa apenas os preços históricos do Yahoo Finance e salva em CSV."""
    arquivo_csv = f"dados_historicos_salvos/{ticker.replace('.', '_')}_historico.csv"
    
    # Tentar carregar dados salvos primeiro
    if os.path.exists(arquivo_csv):
        print(f"\n📂 Carregando dados salvos para {ticker}...")
        try:
            df = pd.read_csv(arquivo_csv, parse_dates=['Date'])
            print(f"✓ {len(df)} registros carregados de arquivo ({df['Date'].min()} a {df['Date'].max()})")
            return df
        except Exception as e:
            print(f"⚠️ Erro ao carregar arquivo salvo: {str(e)}")
            print(f"   Tentando baixar novamente...")
    
    print(f"\n📥 Baixando dados para {ticker}...")
    
    try:
        # Desabilitar verificação SSL no yfinance
        df = yf.download(ticker, start=start_date, end=end_date, progress=False)
        
        if df.empty:
            print(f"❌ Nenhum dado encontrado para {ticker}")
            return None
        
        # Manter apenas o preço de fechamento ajustado
        df = df[['Adj Close']].copy()
        df.columns = ['Close']
        df = df.reset_index()
        
        # Remover NaNs
        df = df.dropna()
        
        # Salvar em CSV para uso futuro
        df.to_csv(arquivo_csv, index=False)
        print(f"✓ {len(df)} registros baixados ({df['Date'].min()} a {df['Date'].max()})")
        print(f"✓ Dados salvos em: {arquivo_csv}")
        
        return df
        
    except Exception as e:
        print(f"❌ Erro ao baixar {ticker}: {str(e)}")
        return None

def criar_sequencias(data, n_steps):
    """Cria sequências de preços para o Transformer."""
    X, y = [], []
    
    for i in range(len(data) - n_steps):
        X.append(data[i:i+n_steps])
        y.append(data[i+n_steps])
    
    return np.array(X), np.array(y)

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    """Bloco Transformer Encoder."""
    # Multi-head attention
    x = layers.MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=dropout
    )(inputs, inputs)
    x = layers.Dropout(dropout)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    res = x + inputs
    
    # Feed Forward
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(res)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    
    return x + res

def build_transformer_model(
    input_shape,
    head_size=256,
    num_heads=4,
    ff_dim=4,
    num_transformer_blocks=4,
    mlp_units=[128],
    mlp_dropout=0.4,
    dropout=0.25,
):
    """Constrói o modelo Transformer."""
    inputs = keras.Input(shape=input_shape)
    x = inputs
    
    # Transformer Blocks
    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)
    
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)
    
    # MLP Head
    for dim in mlp_units:
        x = layers.Dense(dim, activation="relu")(x)
        x = layers.Dropout(mlp_dropout)(x)
    
    outputs = layers.Dense(1)(x)
    
    return keras.Model(inputs, outputs)

def calcular_metricas(y_true, y_pred):
    """Calcula métricas de avaliação."""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    # MAPE (Mean Absolute Percentage Error)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    
    return {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2,
        'MAPE': mape
    }

# =============================================================================
# PROCESSAMENTO PRINCIPAL
# =============================================================================

resultados_globais = []

for ticker in tickers:
    print("\n" + "="*80)
    print(f"🔄 Processando: {ticker}")
    print("="*80)
    
    # Baixar dados
    df = baixar_dados_historicos(ticker)
    
    if df is None or len(df) < 100:
        print(f"⚠️ Dados insuficientes para {ticker}. Pulando...")
        continue
    
    # Separar treino e teste
    df_treino = df[df['Date'] < DATA_CORTE].copy()
    df_teste = df[df['Date'] >= DATA_CORTE].copy()
    
    print(f"\n📊 Divisão dos dados:")
    print(f"   Treino: {len(df_treino)} registros ({df_treino['Date'].min()} a {df_treino['Date'].max()})")
    print(f"   Teste:  {len(df_teste)} registros ({df_teste['Date'].min()} a {df_teste['Date'].max()})")
    
    if len(df_treino) < 100 or len(df_teste) < 20:
        print(f"⚠️ Dados insuficientes após divisão. Pulando...")
        continue
    
    # Normalizar os preços
    scaler = MinMaxScaler(feature_range=(0, 1))
    
    precos_treino = df_treino['Close'].values.reshape(-1, 1)
    precos_teste = df_teste['Close'].values.reshape(-1, 1)
    
    precos_treino_scaled = scaler.fit_transform(precos_treino)
    precos_teste_scaled = scaler.transform(precos_teste)
    
    # Processar para cada janela temporal
    for n_steps in janelas_temporais:
        print(f"\n{'─'*80}")
        print(f"🔧 Janela Temporal: {n_steps} dias")
        print(f"{'─'*80}")
        
        # Criar sequências
        X_train, y_train = criar_sequencias(precos_treino_scaled.flatten(), n_steps)
        X_test, y_test = criar_sequencias(precos_teste_scaled.flatten(), n_steps)
        
        # Reshape para Transformer (samples, timesteps, features)
        X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
        X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))
        
        print(f"   X_train shape: {X_train.shape}")
        print(f"   X_test shape: {X_test.shape}")
        
        # Construir modelo
        model = build_transformer_model(
            input_shape=(n_steps, 1),
            head_size=256,
            num_heads=4,
            ff_dim=4,
            num_transformer_blocks=4,
            mlp_units=[128],
            mlp_dropout=0.4,
            dropout=0.25,
        )
        
        model.compile(
            loss="mean_squared_error",
            optimizer=keras.optimizers.Adam(learning_rate=1e-4),
            metrics=["mae"]
        )
        
        print(f"\n🤖 Modelo Transformer criado!")
        print(f"   Parâmetros: {model.count_params():,}")
        
        # Callbacks
        early_stop = EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        )
        
        reduce_lr = ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        )
        
        # Treinar modelo
        print(f"\n🏋️ Treinando modelo...")
        
        history = model.fit(
            X_train, y_train,
            validation_split=0.2,
            epochs=100,
            batch_size=32,
            callbacks=[early_stop, reduce_lr],
            verbose=1
        )
        
        # Fazer previsões
        y_train_pred = model.predict(X_train, verbose=0)
        y_test_pred = model.predict(X_test, verbose=0)
        
        # Desnormalizar
        y_train_pred_original = scaler.inverse_transform(y_train_pred)
        y_test_pred_original = scaler.inverse_transform(y_test_pred.reshape(-1, 1))
        
        y_train_original = scaler.inverse_transform(y_train.reshape(-1, 1))
        y_test_original = scaler.inverse_transform(y_test.reshape(-1, 1))
        
        # Calcular métricas
        metricas_treino = calcular_metricas(y_train_original.flatten(), y_train_pred_original.flatten())
        metricas_teste = calcular_metricas(y_test_original.flatten(), y_test_pred_original.flatten())
        
        print(f"\n📊 Métricas de Treino:")
        print(f"   RMSE: {metricas_treino['RMSE']:.4f}")
        print(f"   MAE:  {metricas_treino['MAE']:.4f}")
        print(f"   R²:   {metricas_treino['R2']:.4f}")
        print(f"   MAPE: {metricas_treino['MAPE']:.2f}%")
        
        print(f"\n📊 Métricas de Teste:")
        print(f"   RMSE: {metricas_teste['RMSE']:.4f}")
        print(f"   MAE:  {metricas_teste['MAE']:.4f}")
        print(f"   R²:   {metricas_teste['R2']:.4f}")
        print(f"   MAPE: {metricas_teste['MAPE']:.2f}%")
        
        # Salvar resultados
        resultados_globais.append({
            'Ticker': ticker,
            'Janela': f"Janela_{n_steps}",
            'RMSE_Treino': metricas_treino['RMSE'],
            'MAE_Treino': metricas_treino['MAE'],
            'R2_Treino': metricas_treino['R2'],
            'MAPE_Treino': metricas_treino['MAPE'],
            'RMSE_Teste': metricas_teste['RMSE'],
            'MAE_Teste': metricas_teste['MAE'],
            'R2_Teste': metricas_teste['R2'],
            'MAPE_Teste': metricas_teste['MAPE']
        })
        
        # Salvar previsões de teste
        datas_teste_validas = df_teste['Date'].iloc[n_steps:].reset_index(drop=True)
        
        df_previsoes = pd.DataFrame({
            'Data': datas_teste_validas,
            'Preço Real': y_test_original.flatten(),
            'Preço Previsto': y_test_pred_original.flatten()
        })
        
        arquivo_previsoes = f"resultados_transformer_preco/{ticker}_Janela_{n_steps}_previsoes_teste_final.csv"
        df_previsoes.to_csv(arquivo_previsoes, index=False)
        print(f"✓ Previsões salvas: {arquivo_previsoes}")
        
        # Plotar previsões
        plt.figure(figsize=(15, 6))
        plt.plot(datas_teste_validas, y_test_original, label='Real', linewidth=2)
        plt.plot(datas_teste_validas, y_test_pred_original, label='Previsto', linewidth=2, alpha=0.7)
        plt.title(f'{ticker} - Janela {n_steps} dias - Transformer (Apenas Preços)', fontsize=14, fontweight='bold')
        plt.xlabel('Data', fontsize=12)
        plt.ylabel('Preço (R$)', fontsize=12)
        plt.legend(fontsize=10)
        plt.grid(alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        arquivo_grafico = f"resultados_transformer_preco/{ticker}_Janela_{n_steps}_grafico.png"
        plt.savefig(arquivo_grafico, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"✓ Gráfico salvo: {arquivo_grafico}")
        
        # Salvar histórico de treinamento
        plt.figure(figsize=(12, 5))
        
        plt.subplot(1, 2, 1)
        plt.plot(history.history['loss'], label='Treino')
        plt.plot(history.history['val_loss'], label='Validação')
        plt.title('Loss durante Treinamento')
        plt.xlabel('Época')
        plt.ylabel('Loss (MSE)')
        plt.legend()
        plt.grid(alpha=0.3)
        
        plt.subplot(1, 2, 2)
        plt.plot(history.history['mae'], label='Treino')
        plt.plot(history.history['val_mae'], label='Validação')
        plt.title('MAE durante Treinamento')
        plt.xlabel('Época')
        plt.ylabel('MAE')
        plt.legend()
        plt.grid(alpha=0.3)
        
        plt.tight_layout()
        arquivo_historico = f"resultados_transformer_preco/{ticker}_Janela_{n_steps}_historico_treinamento.png"
        plt.savefig(arquivo_historico, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"✓ Histórico salvo: {arquivo_historico}")

# =============================================================================
# CONSOLIDAR RESULTADOS
# =============================================================================

print("\n" + "="*80)
print(" "*25 + "CONSOLIDANDO RESULTADOS")
print("="*80)

df_resultados = pd.DataFrame(resultados_globais)
arquivo_resultados = "resultados_transformer_preco/resultados_consolidados.csv"
df_resultados.to_csv(arquivo_resultados, index=False)
print(f"\n✓ Resultados consolidados salvos: {arquivo_resultados}")

# Resumo por ticker
print("\n📊 Resumo por Ticker (Melhor Janela - Menor RMSE Teste):")
print("─"*80)

if len(df_resultados) > 0:
    resumo = df_resultados.loc[df_resultados.groupby('Ticker')['RMSE_Teste'].idxmin()]
    resumo = resumo.sort_values('RMSE_Teste')

    for _, row in resumo.iterrows():
        print(f"\n{row['Ticker']}:")
        print(f"  Melhor Janela: {row['Janela']}")
        print(f"  RMSE Teste: {row['RMSE_Teste']:.4f}")
        print(f"  MAE Teste:  {row['MAE_Teste']:.4f}")
        print(f"  R² Teste:   {row['R2_Teste']:.4f}")
        print(f"  MAPE Teste: {row['MAPE_Teste']:.2f}%")
else:
    print("\n⚠️ Nenhum resultado foi gerado. Verifique se os dados foram baixados corretamente.")

print("\n" + "="*80)
print(" "*30 + "✅ EXECUÇÃO CONCLUÍDA")
print("="*80)
print(f"\nResultados salvos em: resultados_transformer_preco/")
print("Dados históricos salvos em: dados_historicos_salvos/")
print("\nArquivos gerados:")
print("  - Previsões de teste (CSV)")
print("  - Gráficos de previsões (PNG)")
print("  - Históricos de treinamento (PNG)")
print("  - Resultados consolidados (CSV)")
print("  - Dados históricos dos tickers (CSV)")


                    TRANSFORMER - APENAS PREÇOS HISTÓRICOS
Tickers: BEEF3.SA, BRFS3.SA, VALE3.SA, JBSS3.SA, SOJA3.SA, SUZB3.SA, GGBR3.SA, CSNA3.SA
Janelas Temporais: [5, 10, 15, 20]
Data de Corte Treino/Teste: 2023-01-01


🔄 Processando: BEEF3.SA

📥 Baixando dados para BEEF3.SA...


Failed to get ticker 'BEEF3.SA' reason: Failed to perform, curl: (77) error setting certificate verify locations:  CAfile: c:\Users\leona\OneDrive\Área de Trabalho\Machine-Learning---Stock-Prediction\.venv\Lib\site-packages\certifi\cacert.pem CApath: none. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$BEEF3.SA: possibly delisted; no timezone found

1 Failed download:
['BEEF3.SA']: possibly delisted; no timezone found
Failed to get ticker 'BRFS3.SA' reason: Failed to perform, curl: (77) error setting certificate verify locations:  CAfile: c:\Users\leona\OneDrive\Área de Trabalho\Machine-Learning---Stock-Prediction\.venv\Lib\site-packages\certifi\cacert.pem CApath: none. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$BRFS3.SA: possibly delisted; no timezone found

1 Failed download:
['BRFS3.SA']: possibly delisted; no timezone found


❌ Nenhum dado encontrado para BEEF3.SA
⚠️ Dados insuficientes para BEEF3.SA. Pulando...

🔄 Processando: BRFS3.SA

📥 Baixando dados para BRFS3.SA...
❌ Nenhum dado encontrado para BRFS3.SA
⚠️ Dados insuficientes para BRFS3.SA. Pulando...

🔄 Processando: VALE3.SA

📥 Baixando dados para VALE3.SA...


Failed to get ticker 'VALE3.SA' reason: Failed to perform, curl: (77) error setting certificate verify locations:  CAfile: c:\Users\leona\OneDrive\Área de Trabalho\Machine-Learning---Stock-Prediction\.venv\Lib\site-packages\certifi\cacert.pem CApath: none. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$VALE3.SA: possibly delisted; no timezone found

1 Failed download:
['VALE3.SA']: possibly delisted; no timezone found
Failed to get ticker 'JBSS3.SA' reason: Failed to perform, curl: (77) error setting certificate verify locations:  CAfile: c:\Users\leona\OneDrive\Área de Trabalho\Machine-Learning---Stock-Prediction\.venv\Lib\site-packages\certifi\cacert.pem CApath: none. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$JBSS3.SA: possibly delisted; no timezone found

1 Failed download:
['JBSS3.SA']: possibly delisted; no timezone found
Failed to get ticker 'SOJA3.SA' reason: Failed to perform, curl: (77) error setting certificate veri

❌ Nenhum dado encontrado para VALE3.SA
⚠️ Dados insuficientes para VALE3.SA. Pulando...

🔄 Processando: JBSS3.SA

📥 Baixando dados para JBSS3.SA...
❌ Nenhum dado encontrado para JBSS3.SA
⚠️ Dados insuficientes para JBSS3.SA. Pulando...

🔄 Processando: SOJA3.SA

📥 Baixando dados para SOJA3.SA...


$SOJA3.SA: possibly delisted; no timezone found

1 Failed download:
['SOJA3.SA']: possibly delisted; no timezone found
Failed to get ticker 'SUZB3.SA' reason: Failed to perform, curl: (77) error setting certificate verify locations:  CAfile: c:\Users\leona\OneDrive\Área de Trabalho\Machine-Learning---Stock-Prediction\.venv\Lib\site-packages\certifi\cacert.pem CApath: none. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$SUZB3.SA: possibly delisted; no timezone found

1 Failed download:
['SUZB3.SA']: possibly delisted; no timezone found
Failed to get ticker 'GGBR3.SA' reason: Failed to perform, curl: (77) error setting certificate verify locations:  CAfile: c:\Users\leona\OneDrive\Área de Trabalho\Machine-Learning---Stock-Prediction\.venv\Lib\site-packages\certifi\cacert.pem CApath: none. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$GGBR3.SA: possibly delisted; no timezone found

1 Failed download:
['GGBR3.SA']: possibly delisted;

❌ Nenhum dado encontrado para SOJA3.SA
⚠️ Dados insuficientes para SOJA3.SA. Pulando...

🔄 Processando: SUZB3.SA

📥 Baixando dados para SUZB3.SA...
❌ Nenhum dado encontrado para SUZB3.SA
⚠️ Dados insuficientes para SUZB3.SA. Pulando...

🔄 Processando: GGBR3.SA

📥 Baixando dados para GGBR3.SA...
❌ Nenhum dado encontrado para GGBR3.SA
⚠️ Dados insuficientes para GGBR3.SA. Pulando...

🔄 Processando: CSNA3.SA

📥 Baixando dados para CSNA3.SA...
❌ Nenhum dado encontrado para CSNA3.SA
⚠️ Dados insuficientes para CSNA3.SA. Pulando...

                         CONSOLIDANDO RESULTADOS

✓ Resultados consolidados salvos: resultados_transformer_preco/resultados_consolidados.csv

📊 Resumo por Ticker (Melhor Janela - Menor RMSE Teste):
────────────────────────────────────────────────────────────────────────────────

⚠️ Nenhum resultado foi gerado. Verifique se os dados foram baixados corretamente.

                              ✅ EXECUÇÃO CONCLUÍDA

Resultados salvos em: resultados_transformer_preco/

In [12]:
# Execute isso no ambiente GLOBAL (Python 3.14)
import pandas as pd
import os

pasta_origem = r"C:\Users\leona\OneDrive\Área de Trabalho\Machine-Learning---Stock-Prediction\CodigoExperimentos\ExperimentoFeatures\dataframesMicro"
pasta_destino = r"C:\Users\leona\OneDrive\Área de Trabalho\Machine-Learning---Stock-Prediction\CodigoExperimentos\ExperimentoFeatures\dados_preco"

os.makedirs(pasta_destino, exist_ok=True)

# Arquivos para extrair
arquivos = {
    "JBSS3.SA": "JBSS3.SA_dados_unificados.csv",
}

print("="*80)
print("         EXTRAINDO DADOS DE PREÇO DOS ARQUIVOS EXISTENTES")
print("="*80 + "\n")

for ticker, nome_arquivo in arquivos.items():
    arquivo_origem = os.path.join(pasta_origem, nome_arquivo)
    
    if not os.path.exists(arquivo_origem):
        print(f"⚠️ Arquivo não encontrado: {nome_arquivo}\n")
        continue
    
    print(f"📂 Processando {ticker}...")
    
    try:
        # Ler arquivo com primeira coluna como índice
        df = pd.read_csv(arquivo_origem, index_col=0)
        
        print(f"  Colunas disponíveis: {df.columns.tolist()}")
        print(f"  Índice (Datas): {df.index.name if df.index.name else 'sem nome'}")
        
        # Verificar se tem a coluna Preço
        if 'Preço' not in df.columns:
            print(f"  ❌ Coluna 'Preço' não encontrada\n")
            continue
        
        # Criar DataFrame com Date e Close
        df_preco = pd.DataFrame({
            'Date': pd.to_datetime(df.index),
            'Close': df['Preço'].values
        })
        
        # Filtrar período 2020-2024
        df_preco = df_preco[(df_preco['Date'] >= '2020-01-01') & (df_preco['Date'] <= '2024-12-31')]
        
        # Remover NaNs
        df_preco = df_preco.dropna()
        
        # Ordenar por data
        df_preco = df_preco.sort_values('Date').reset_index(drop=True)
        
        if len(df_preco) == 0:
            print(f"  ❌ Sem dados válidos no período 2020-2024\n")
            continue
        
        # Salvar
        arquivo_destino = os.path.join(pasta_destino, f"{ticker.replace('.', '_')}_historico.csv")
        df_preco.to_csv(arquivo_destino, index=False)
        
        print(f"  ✓ Salvo: {arquivo_destino}")
        print(f"  Registros: {len(df_preco)} | Período: {df_preco['Date'].min()} a {df_preco['Date'].max()}\n")
        
    except Exception as e:
        print(f"  ❌ Erro: {str(e)}\n")
        import traceback
        traceback.print_exc()

print("="*80)
print("                    ✅ CONCLUÍDO")
print("="*80)

         EXTRAINDO DADOS DE PREÇO DOS ARQUIVOS EXISTENTES

📂 Processando JBSS3.SA...
  Colunas disponíveis: ['Preço', 'ROA', 'ROE', 'Margem Líquida', 'P/L', 'VP', 'Preço_anterior']
  Índice (Datas): sem nome
  ✓ Salvo: C:\Users\leona\OneDrive\Área de Trabalho\Machine-Learning---Stock-Prediction\CodigoExperimentos\ExperimentoFeatures\dados_preco\JBSS3_SA_historico.csv
  Registros: 1460 | Período: 2020-01-02 00:00:00 a 2023-12-31 00:00:00

                    ✅ CONCLUÍDO
